# GEOCK + BindingDB: Train on 1M+ compounds with GPU XGBoost

This notebook downloads BindingDB (1M compounds), computes 2D fingerprints (ECFP4+MACCS+FCFP4), deduplicates against Phase 2 training data, and trains an XGBoost model on GPU for binding affinity prediction.

In [ ]:
# Mount Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/geock_bindingdb'
import os; os.makedirs(BASE, exist_ok=True)
%cd $BASE

In [ ]:
# Install dependencies
!apt-get update -qq && apt-get install -y -qq libre2-dev libssl-dev
!pip install -q rdkit-pypi xgboost scikit-learn pandas pyarrow huggingface-hub scipy

import numpy as np
import pandas as pd
import pickle
import time
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
import xgboost as xgb

print(f'XGBoost version: {xgb.__version__}')
print(f'GPU available: {xgb.core._have_gpu_support}')
print(f'Num GPUs: {xgb.core._get_gpu_count() if hasattr(xgb.core, "_get_gpu_count") else "check below"}')
!nvidia-smi | head -5

In [ ]:
# 1. Download BindingDB from HuggingFace (~375 MB)
from huggingface_hub import hf_hub_download
import os

print('Downloading BindingDB parquet files...')
files = [
    'data/train-00000-of-00002.parquet',
    'data/train-00001-of-00002.parquet',
]
for f in files:
    local_name = f.replace('/', '_')
    local_path = f'{BASE}/{local_name}'
    if os.path.exists(local_path):
        print(f'  {local_name}: exists ({os.path.getsize(local_path)/1e6:.0f} MB)')
        continue
    print(f'  Downloading {f}...')
    t0 = time.time()
    path = hf_hub_download('vladak/bindingdb', f, repo_type='dataset')
    import shutil; shutil.copy2(path, local_path)
    print(f'    done ({time.time()-t0:.0f}s, {os.path.getsize(local_path)/1e6:.0f} MB)')

In [ ]:
# 2. Compute 982-dim fingerprints in parallel
from multiprocessing import Pool
from functools import partial

def compute_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    ecfp = AllChem.GetMorganFingerprintAsBitVect(mol, 4, nBits=512)
    maccs = MACCSkeys.GenMACCSKeys(mol)
    fcfp = AllChem.GetMorganFingerprintAsBitVect(mol, 4, nBits=300, useFeatures=True)
    fp = np.zeros(982, dtype=np.uint8)
    fp[:512] = np.array(ecfp)
    fp[512:679] = np.array(maccs)
    fp[679:979] = np.array(fcfp)
    return fp

def process_chunk(args):
    smiles_list, ic50_list = args
    fps, ys = [], []
    for smi, ic50 in zip(smiles_list, ic50_list):
        if np.isnan(ic50): continue
        fp = compute_fp(smi)
        if fp is not None:
            fps.append(fp); ys.append(ic50)
    return np.array(fps, dtype=np.uint8), np.array(ys, dtype=np.float64)

import pyarrow.parquet as pq

t0 = time.time()
print('Reading parquet files...')
dfs = []
for i in range(2):
    df = pd.read_parquet(f'{BASE}/data_train-0000{i}-of-00002.parquet')
    dfs.append(df)
    print(f'  File {i}: {len(df)} entries')
df_all = pd.concat(dfs, ignore_index=True)
print(f'Total: {len(df_all)} entries ({time.time()-t0:.0f}s)')

# Split into chunks for parallel processing
n_cores = os.cpu_count()
chunk_size = len(df_all) // (n_cores * 4)
chunks = [(df_all.iloc[i:i+chunk_size]['ligand'].tolist(),
           df_all.iloc[i:i+chunk_size]['ic50'].tolist())
          for i in range(0, len(df_all), chunk_size)]
print(f'Processing {len(chunks)} chunks with {n_cores} cores...')

t1 = time.time()
with Pool(n_cores) as pool:
    results = pool.map(process_chunk, chunks)

X_bind = np.concatenate([r[0] for r in results])
y_bind = np.concatenate([r[1] for r in results])
print(f'Fingerprints: {X_bind.shape} ({time.time()-t1:.0f}s)')
print(f'y: mean={y_bind.mean():.3f}, std={y_bind.std():.3f}')

np.save(f'{BASE}/X_bindingdb.npy', X_bind)
np.save(f'{BASE}/y_bindingdb.npy', y_bind)
print('Saved to Google Drive')

In [ ]:
# 3. Download Phase 2 training data from GitHub
print('Downloading Phase 2 training data...')
!wget -q https://github.com/YOUR_USERNAME/geock/raw/main/autoresearch_backup/phase2_X.npy -O $BASE/phase2_X.npy
!wget -q https://github.com/YOUR_USERNAME/geock/raw/main/autoresearch_backup/phase2_y.npy -O $BASE/phase2_y.npy

import numpy as np
X_phase2 = np.load(f'{BASE}/phase2_X.npy')
y_phase2 = np.load(f'{BASE}/phase2_y.npy')
print(f'Phase 2: {X_phase2.shape}, y: mean={y_phase2.mean():.3f}, std={y_phase2.std():.3f}')

In [ ]:
# 4. Deduplicate BindingDB against Phase 2
print('Deduplicating...')
p2_hashes = set(hash(row.tobytes()) for row in X_phase2[:, :512])
print(f'Phase 2 unique molecules: {len(p2_hashes)}')

new_mask = np.ones(len(X_bind), dtype=bool)
unique_h = set()
for i in range(len(X_bind)):
    h = hash(X_bind[i, :512].tobytes())
    if h in p2_hashes or h in unique_h:
        new_mask[i] = False
    else:
        unique_h.add(h)

X_new = X_bind[new_mask]
y_new = y_bind[new_mask]
print(f'Dups removed: {len(X_bind) - len(X_new)}')
print(f'New molecules: {len(X_new)}')

In [ ]:
# 5. Train with XGBoost GPU
# Combine Phase 2 + BindingDB (zero-pad to 1032 for compatibility)
X_bind_aug = np.zeros((len(X_new), 1032), dtype=np.float32)
X_bind_aug[:, :982] = X_new.astype(np.float32)
X_phase2_aug = np.zeros((len(X_phase2), 1032), dtype=np.float32)
X_phase2_aug[:, :982] = X_phase2.astype(np.float32)

X_all = np.concatenate([X_phase2_aug, X_bind_aug])
y_all = np.concatenate([y_phase2, y_new])
print(f'Training data: {X_all.shape}')

# Standardize and select features
ss = StandardScaler()
X_s = ss.fit_transform(X_all)
sel = SelectKBest(f_regression, k=500)
X_sel = sel.fit_transform(X_s, y_all)

# XGBoost with GPU
model = xgb.XGBRegressor(
    max_depth=12, n_estimators=2000, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, gamma=0.1,
    reg_alpha=0.5, reg_lambda=2.0,
    tree_method='gpu_hist', predictor='gpu_predictor',
    random_state=42, n_jobs=-1, verbosity=1
)

t0 = time.time()
model.fit(X_sel, y_all)
train_time = time.time() - t0
print(f'Training: {train_time:.0f}s ({train_time/60:.1f} min)')

In [ ]:
# 6. Evaluate on CASF-2016
# Load CASF-2016 reference (PDB ID + pKd + SMILES)
# Note: You'll need the casf2016_reference.csv file
# If Colab can't access it, download from GitHub

import urllib.request
url = 'https://raw.githubusercontent.com/YOUR_USERNAME/geock/main/autoresearch_backup/casf2016_reference.csv'
urllib.request.urlretrieve(url, f'{BASE}/casf2016_reference.csv')

casf_df = pd.read_csv(f'{BASE}/casf2016_reference.csv')
print(f'CASF-2016 complexes: {len(casf_df)}')

true_vals, pred_vals = [], []
for _, row in casf_df.iterrows():
    smiles = row['smiles']
    if pd.isna(smiles) or not smiles:
        continue
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue
    fp = compute_fp(smiles)
    if fp is None:
        continue
    X_test = np.zeros(1032, dtype=np.float32)
    X_test[:982] = fp.astype(np.float32)
    X_test_s = ss.transform(X_test.reshape(1, -1))
    X_test_sel = sel.transform(X_test_s)
    pred = model.predict(X_test_sel)[0]
    true_vals.append(row['pkd_true'])
    pred_vals.append(pred)

true_arr = np.array(true_vals)
pred_arr = np.array(pred_vals)
r_p, _ = pearsonr(true_arr, pred_arr)
r_s, _ = spearmanr(true_arr, pred_arr)
rmse = np.sqrt(np.mean((true_arr - pred_arr)**2))
mae = np.mean(np.abs(true_arr - pred_arr))

print(f'\n=== CASF-2016 Results ({len(true_vals)} complexes) ===')
print(f'Pearson R: {r_p:.4f}')
print(f'Spearman R: {r_s:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAE: {mae:.4f}')

In [ ]:
# 7. Save model
model_data = {
    'model': model,
    'scaler': ss,
    'selector': sel,
    'config': 'BindingDB+Phase2, k=500, t=2000, GPU',
    'n_samples': len(X_all),
    'casf2016_r': float(r_p)
}
with open(f'{BASE}/geock_bindingdb_final.pkl', 'wb') as f:
    pickle.dump(model_data, f)
print(f'Model saved: {os.path.getsize(f"{BASE}/geock_bindingdb_final.pkl")/1e6:.0f} MB')
print(f'\n=== Comparison with Best Previous ===')
print(f'Phase 5c (19K only):     CASF-2016 R = 0.731')
print(f'BindingDB (556K+19K):    CASF-2016 R = {r_p:.4f}')